# Parte 2 — Visualización de datos

**Objetivo:** Explorar visualmente el datamart y comunicar insights de negocio mediante gráficos en Python (complemento al tablero de Power BI).

## Visualizaciones incluidas

1. Evolución mensual de ventas
2. Curva de Pareto de productos
3. Ventas por categoría
4. Mapa de calor mes × categoría

Las figuras se exportan a `informe/img/` para el informe consolidado.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
DATA = PROJECT_ROOT / "data" / "processed"
IMG_DIR = PROJECT_ROOT / "informe" / "img"
IMG_DIR.mkdir(parents=True, exist_ok=True)

SEP = ";"
ENC = "utf-8"
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

print("Imágenes en: informe/img")

In [ ]:
clientes = pd.read_csv(DATA / "dim_cliente.csv", sep=SEP, encoding=ENC)
productos = pd.read_csv(DATA / "dim_producto.csv", sep=SEP, encoding=ENC)
tiendas = pd.read_csv(DATA / "dim_tienda.csv", sep=SEP, encoding=ENC)
ventas = pd.read_csv(DATA / "fact_ventas.csv", sep=SEP, encoding=ENC)

ventas["fecha"] = pd.to_datetime(ventas["fecha"])

df = (
    ventas
    .merge(clientes[["id_cliente", "segmento_programa", "region"]], on="id_cliente", how="left")
    .merge(productos[["id_producto", "nombre", "categoria", "producto_estrella"]], on="id_producto", how="left")
    .merge(tiendas[["id_tienda", "canal", "region"]], on="id_tienda", how="left", suffixes=("_cliente", "_tienda"))
)

print(f"Registros analizados: {len(df):,}")
print(f"Ventas totales: S/ {df['importe_venta'].sum():,.2f}")
print(f"Margen total: S/ {df['margen'].sum():,.2f}")
print(f"Ticket promedio: S/ {df.groupby('id_venta')['importe_venta'].sum().mean():,.2f}")

In [ ]:
# 1) Evolución mensual de ventas
ventas_mes = (
    df.groupby(df["fecha"].dt.to_period("M"))["importe_venta"]
    .sum()
    .sort_index()
)
ventas_mes.index = ventas_mes.index.to_timestamp()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ventas_mes.index, ventas_mes.values, marker="o", linewidth=2)
ax.set_title("Evolución mensual de ventas — Retail ficticio")
ax.set_xlabel("Mes")
ax.set_ylabel("Ventas (S/)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(IMG_DIR / "ventas_mensuales.png", bbox_inches="tight")
plt.show()
print("Guardado: ventas_mensuales.png")

### Insight 1 — Evolución temporal

El gráfico muestra la tendencia de ingresos mes a mes durante 2024-2025. Permite detectar estacionalidad (picos en meses de mayor consumo) y evaluar si la operación crece o se mantiene estable.

**Recomendación:** usar esta tendencia para planificar inventario y campañas en los meses de mayor volumen.

In [ ]:
# 2) Pareto de productos (80/20)
top_prod = (
    df.groupby(["id_producto", "nombre"])["importe_venta"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
top_prod["pct_acum"] = top_prod["importe_venta"].cumsum() / top_prod["importe_venta"].sum() * 100
top_prod["rank"] = range(1, len(top_prod) + 1)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(top_prod["rank"].head(30), top_prod["importe_venta"].head(30), color="steelblue")
ax1.set_xlabel("Top 30 productos (rank)")
ax1.set_ylabel("Ingresos (S/)")
ax2 = ax1.twinx()
ax2.plot(top_prod["rank"].head(30), top_prod["pct_acum"].head(30), color="darkorange", marker="o")
ax2.axhline(80, color="red", linestyle="--", alpha=0.7, label="80%")
ax2.set_ylabel("% acumulado")
ax1.set_title("Pareto de productos — concentración de ventas")
plt.tight_layout()
fig.savefig(IMG_DIR / "pareto_productos.png", bbox_inches="tight")
plt.show()

n_80 = (top_prod["pct_acum"] <= 80).sum()
print(f"Productos que explican ~80% de ventas: {n_80} ({n_80/len(top_prod)*100:.1f}% del catálogo)")

### Insight 2 — Concentración Pareto

Un subconjunto reducido de productos concentra la mayor parte de los ingresos (regla 80/20). Esto confirma la lógica de **productos estrella** del dataset.

**Recomendación:** asegurar stock y visibilidad de los productos top; evaluar margen individual para no depender solo de volumen.

In [ ]:
# 3) Ventas por categoría
cat = (
    df.groupby("categoria")["importe_venta"]
    .sum()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
cat.plot(kind="barh", ax=ax, color=sns.color_palette("viridis", len(cat)))
ax.set_title("Ventas totales por categoría")
ax.set_xlabel("Ingresos (S/)")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(IMG_DIR / "ventas_por_categoria.png", bbox_inches="tight")
plt.show()
print("Top categoría:", cat.index[-1], f"S/ {cat.iloc[-1]:,.2f}")

### Insight 3 — Categorías líderes

Las categorías con mayor venta definen el mix comercial del negocio. Identificar las líderes ayuda a priorizar abastecimiento y promociones.

**Recomendación:** cruzar categorías líderes con margen en Power BI para distinguir volumen vs rentabilidad.

In [ ]:
# 4) Heatmap mes x categoría
df["mes_periodo"] = df["fecha"].dt.to_period("M").astype(str)
pivot = df.pivot_table(
    index="categoria",
    columns="mes_periodo",
    values="importe_venta",
    aggfunc="sum",
    fill_value=0,
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, cmap="YlOrRd", ax=ax, linewidths=0.3)
ax.set_title("Mapa de calor — ventas por categoría y mes")
ax.set_xlabel("Mes")
ax.set_ylabel("Categoría")
plt.tight_layout()
fig.savefig(IMG_DIR / "heatmap_mes_categoria.png", bbox_inches="tight")
plt.show()

### Insight 4 — Estacionalidad por categoría

El heatmap revela qué categorías venden más en determinados meses, útil para campañas estacionales.

**Recomendación:** alinear promociones (`dim_promocion`) con los meses de mayor intensidad por categoría.

In [ ]:
# 5) KPI adicionales para el informe
segmento = (
    df.groupby("segmento_programa")
    .agg(
        clientes=("id_cliente", "nunique"),
        ingresos=("importe_venta", "sum"),
        margen=("margen", "sum"),
    )
    .sort_values("ingresos", ascending=False)
)
print("=== Ingresos por segmento de fidelización ===")
print(segmento.to_string())

canal = df.groupby("canal")["importe_venta"].sum().sort_values(ascending=False)
print("\n=== Ventas por canal ===")
print(canal.to_string())

insights = pd.DataFrame([
    {"hallazgo": "Ventas totales del periodo", "valor": f"S/ {df['importe_venta'].sum():,.2f}"},
    {"hallazgo": "Margen total", "valor": f"S/ {df['margen'].sum():,.2f}"},
    {"hallazgo": "Ticket promedio", "valor": f"S/ {df.groupby('id_venta')['importe_venta'].sum().mean():,.2f}"},
    {"hallazgo": "Líneas promedio por ticket", "valor": f"{len(df) / df['id_venta'].nunique():.2f}"},
    {"hallazgo": "Segmento con mayor ingreso", "valor": segmento.index[0]},
    {"hallazgo": "Canal con mayor ingreso", "valor": canal.index[0]},
])
display(insights)

## Hallazgos accionables (resumen Parte 2)

| # | Hallazgo | Recomendación |
| --- | --- | --- |
| 1 | Existe tendencia temporal identificable en ventas mensuales | Planificar inventario según picos estacionales |
| 2 | Pocos productos concentran ~80% de ingresos | Proteger disponibilidad de productos estrella |
| 3 | Categorías líderes dominan el mix comercial | Priorizar abastecimiento y exhibición |
| 4 | Patrones distintos por categoría y mes | Diseñar promociones estacionales focalizadas |
| 5 | Segmento Bronce aporta mayor ingreso total | Estrategias de upselling hacia Plata/Oro |
| 6 | Canal físico vs online difiere en contribución | Optimizar experiencia en el canal dominante |
| 7 | Ticket promedio estable permite medir impacto de campañas | Usar ticket promedio como KPI en Power BI |
| 8 | Canasta promedio >1 línea habilita venta cruzada | Complementar con reglas de asociación (Parte 5) |

**Siguiente paso:** replicar estos KPI en Power BI con medidas DAX (`powerbi/medidas_dax.dax`).